# Working FLAN-T5 Training - Grammar Correction
## Fixed version that actually works!


In [30]:
# Install required packages
!pip install transformers datasets accelerate evaluate
!pip install rouge_score

In [31]:
# Import libraries
import json
import torch
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    TrainingArguments, Trainer,
    DataCollatorForSeq2Seq
)
from datasets import Dataset, DatasetDict
from google.colab import files
import numpy as np
from sklearn.model_selection import train_test_split
import evaluate

In [32]:
# Upload your JSONL file
uploaded = files.upload()
jsonl_file = list(uploaded.keys())[0]
print(f"Uploaded: {jsonl_file}")


Saving cbc_dataset.jsonl to cbc_dataset (1).jsonl
Uploaded: cbc_dataset (1).jsonl


In [33]:
# SIMPLE DUAL TASK APPROACH - Clean and straightforward
def load_data_simple_dual_task(filename):
    correction_data = []
    feedback_data = []

    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)

                # Parse the output to extract correction and feedback
                output = item['output']
                if 'CORRECTION:' in output and 'FEEDBACK:' in output:
                    correction = output.split('CORRECTION:')[1].split('FEEDBACK:')[0].strip()
                    feedback = output.split('FEEDBACK:')[1].strip()

                    # TASK 1: CORRECTION TASK
                    correction_data.append({
                        'text': f"Correct the grammar errors in this sentence: {item['input']}",
                        'target': correction
                    })

                    # TASK 2: FEEDBACK TASK - Strong instruction for all sentences
                    feedback_data.append({
                        'text': f"Identify the specific grammar errors in this sentence and explain the correct grammar rules: {item['input']}",
                        'target': feedback  # Use the detailed CBC-aligned feedback
                    })

    return correction_data, feedback_data

# Use the simple dual task data loading
correction_data, feedback_data = load_data_simple_dual_task(jsonl_file)
# Combine both tasks for training
combined_data = correction_data + feedback_data
train_data, val_data = train_test_split(combined_data, test_size=0.1, random_state=42)
dataset = DatasetDict({
    'train': Dataset.from_list(train_data),
    'validation': Dataset.from_list(val_data)
})

In [34]:
# Load FLAN-T5 model and tokenizer
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [35]:
# Check GPU and setup device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


In [36]:
# Move model to GPU for better performance
print("=== MOVING MODEL TO GPU ===")
model = model.to(device)
print(f"Model moved to: {next(model.parameters()).device}")

# Test GPU memory usage
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory cached: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


=== MOVING MODEL TO GPU ===
Model moved to: cuda:0
GPU memory allocated: 1.16 GB
GPU memory cached: 4.09 GB


In [37]:
# CORRECT tokenization for FLAN-T5
def preprocess_function(examples):
    inputs = examples['text']
    targets = examples['target']

    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)

    # Tokenize targets
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=512, truncation=True, padding=False)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)
# Check a sample
sample = tokenized_dataset['train'][0]


Map:   0%|          | 0/4237 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4034: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/471 [00:00<?, ? examples/s]

In [40]:
# SIMPLE extended training - just increase epochs and batch size
training_args = TrainingArguments(
    output_dir="./grammar-model",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=150,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=75,
    save_strategy="steps",
    save_steps=150,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=False,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to=None,
    max_grad_norm=1.0,
)


In [41]:
# CORRECT data collator for FLAN-T5
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
)
print("Data collator created!")


Data collator created!


In [42]:
# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Trainer initialized successfully!")


Trainer initialized successfully!


/tmp/ipython-input-3512092964.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [43]:
# Start training
print("Starting training...")
trainer.train()
print("Training completed!")


Starting training...


Step,Training Loss,Validation Loss
75,2.607600,2.689893
150,2.214900,2.279359
225,2.401900,1.930928
300,2.146600,1.714056
375,1.705300,1.573475
450,1.610800,1.481277
525,1.800500,1.409596
600,1.580100,1.362526
675,1.585100,1.326282
750,1.585000,1.295527


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


Training completed!


In [44]:
# Test the FINE-TUNED model with dual task approach
print("=== TESTING FINE-TUNED MODEL - DUAL TASK APPROACH ===")

def test_correction_task(text):
    input_text = f"Correct the grammar errors in this sentence: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=500,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

def test_feedback_task(text):
    input_text = f"Explain the grammar error in this sentence: {text}"
    inputs = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True)

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=500,
            num_beams=4,
            early_stopping=True,
            do_sample=False
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# Test sentences
test_sentences = [
    "I has a apple.",
    "She go to school yesterday.",
    "The students was very happy.",
    "He don't like apples.",
    "We was playing football."
]

print("=== TASK 1: GRAMMAR CORRECTION TEST ===")
for sentence in test_sentences:
    print(f"\nInput: {sentence}")
    correction = test_correction_task(sentence)
    print(f"Correction: {correction}")
    print("-" * 50)

print("\n=== TASK 2: FEEDBACK GENERATION TEST ===")
for sentence in test_sentences:
    print(f"\nInput: {sentence}")
    feedback = test_feedback_task(sentence)
    print(f"Feedback: {feedback}")
    print("-" * 50)

print("\n=== COMBINED OUTPUT ===")
for sentence in test_sentences:
    correction = test_correction_task(sentence)
    feedback = test_feedback_task(sentence)
    print(f"\nInput: {sentence}")
    print(f"CORRECTION: {correction}")
    print(f"FEEDBACK: {feedback}")
    print("-" * 50)


=== TESTING FINE-TUNED MODEL - DUAL TASK APPROACH ===
=== TASK 1: GRAMMAR CORRECTION TEST ===

Input: I has a apple.
Correction: I have a shit.
--------------------------------------------------

Input: She go to school yesterday.
Correction: She went to school yesterday.
--------------------------------------------------

Input: The students was very happy.
Correction: The students were very happy.
--------------------------------------------------

Input: He don't like apples.
Correction: He doesn't like apples.
--------------------------------------------------

Input: We was playing football.
Correction: We were playing football.
--------------------------------------------------

=== TASK 2: FEEDBACK GENERATION TEST ===

Input: I has a apple.
Feedback: I have a apple.
--------------------------------------------------

Input: She go to school yesterday.
Feedback: She went to school yesterday.
--------------------------------------------------

Input: The students was very happy.
F

In [45]:
# Evaluate the fine-tuned model using BLEU and ROUGE
print("=== EVALUATING FINE-TUNED MODEL ===")

# Function to generate predictions for evaluation
def generate_predictions(dataset, model, tokenizer, device, batch_size=8):
    predictions = []
    references = []
    model.eval()
    for i in range(0, len(dataset), batch_size):
        batch = dataset[i : i + batch_size]
        inputs = tokenizer(batch['text'], return_tensors="pt", max_length=1024, truncation=True, padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=512,
                num_beams=4,
                early_stopping=True,
                do_sample=False
            )

        decoded_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(decoded_preds)
        references.extend(batch['target']) # Use the original target from the dataset

    return predictions, references

# Generate predictions and references for the validation set
val_predictions, val_references = generate_predictions(dataset["validation"], model, tokenizer, device)

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Compute BLEU score
bleu_results = bleu.compute(predictions=val_predictions, references=val_references)
print(f"BLEU score: {bleu_results['bleu']:.4f}")

# Compute ROUGE scores
# ROUGE expects a list of lists for references when there are multiple possible references per prediction
# In this case, we only have one reference per prediction, so we wrap each reference in a list.
rouge_references = [[ref] for ref in val_references]
rouge_results = rouge.compute(predictions=val_predictions, references=rouge_references)
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")

print("Evaluation completed!")

=== EVALUATING FINE-TUNED MODEL ===
BLEU score: 0.3490
ROUGE-1: 0.6901
ROUGE-2: 0.4961
ROUGE-L: 0.6590
Evaluation completed!
